# Create multi-agent research tool using Autogen

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

from typing import List, Sequence
from autogen_agentchat.agents import AssistantAgent, UserProxyAgent
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.messages import BaseAgentEvent, BaseChatMessage
from autogen_agentchat.teams import SelectorGroupChat, RoundRobinGroupChat
from autogen_agentchat.ui import Console
import requests
import xml.etree.ElementTree as ET



In [2]:
from autogen_ext.models.openai import OpenAIChatCompletionClient

client = OpenAIChatCompletionClient(model="gpt-4o")

Implement an arXiv search function

In [3]:
def arxiv_search_tool(query: str, max_results: int) -> str:
    """
    Search for papers on arXiv based on the given query.

    Args:
        query (str): The search query.
        max_results (int): The maximum number of results to return.

    Returns:
        str: A list of the top results.
    """
    # Implementation for arXiv search would go here
    url = f'http://export.arxiv.org/api/query'
    response = requests.get(url, params={'search_query': query, 'max_results': max_results})

    if response.status_code == 200:
        root = ET.fromstring(response.content)
        entries = root.findall('{http://www.w3.org/2005/Atom}entry')
        results = []
        for entry in entries:
            title = entry.find('{http://www.w3.org/2005/Atom}title').text
            summary = entry.find('{http://www.w3.org/2005/Atom}summary').text
            results.append(f"Title: {title}\nSummary: {summary}\n")
        return "\n".join(results)
    else:
        return f"Error: Unable to fetch results from arXiv. Status code: {response.status_code}"
    

In [ ]:
task = "AI in healthcare"
result = arxiv_search_tool(task, 3)
print(result)

Title: An Overview and Case Study of the Clinical AI Model Development Life Cycle for Healthcare Systems
Summary: Healthcare is one of the most promising areas for machine learning models to make a positive impact. However, successful adoption of AI-based systems in healthcare depends on engaging and educating stakeholders from diverse backgrounds about the development process of AI models. We present a broadly accessible overview of the development life cycle of clinical AI models that is general enough to be adapted to most machine learning projects, and then give an in-depth case study of the development process of a deep learning based system to detect aortic aneurysms in Computed Tomography (CT) exams. We hope other healthcare institutions and clinical practitioners find the insights we share about the development process useful in informing their own model development efforts and to increase the likelihood of successful deployment and integration of AI in healthcare.

Title: Fait

Create a Topic refinement agent

In [5]:
agent_description = "Refines a broad research idea into a specific topic."
SYSTEM_MESSAGE = """Role: You are a research topic refinement agent.
Task: Given a generic idea, output one refined research topic.
Output_format: 
  Refined Research Topic: <refined topic>
  End with TASK_COMPLETED_TOPIC_AGENT
"""
tools = [arxiv_search_tool]
topicAgent = AssistantAgent(name="TopicAgent",
                            description = agent_description,
                            system_message = SYSTEM_MESSAGE,
                            tools = tools,
                            model_client = client
                            )

In [ ]:
topic_result = await topicAgent.run(task=task)
print(topic_result.messages[-1].content)


Title: An Overview and Case Study of the Clinical AI Model Development Life Cycle for Healthcare Systems
Summary: Healthcare is one of the most promising areas for machine learning models to make a positive impact. However, successful adoption of AI-based systems in healthcare depends on engaging and educating stakeholders from diverse backgrounds about the development process of AI models. We present a broadly accessible overview of the development life cycle of clinical AI models that is general enough to be adapted to most machine learning projects, and then give an in-depth case study of the development process of a deep learning based system to detect aortic aneurysms in Computed Tomography (CT) exams. We hope other healthcare institutions and clinical practitioners find the insights we share about the development process useful in informing their own model development efforts and to increase the likelihood of successful deployment and integration of AI in healthcare.

Title: Fait

Create a Paper discovery agent

In [6]:
agent_description = "Searches for relevant papers based on a research topic."
SYSTEM_MESSAGE = """Role: You are a research paper search agent.
Task: Given a research topic, search for relevant papers and return the top results.
"""
tools = [arxiv_search_tool]
paperAgent = AssistantAgent(name="PaperAgent",
                            description = agent_description,
                            system_message = SYSTEM_MESSAGE,
                            tools = tools,
                            model_client = client
                            )


In [ ]:
task = "AI in healthcare"
paper_result = await paperAgent.run(task=task)
print(paper_result.messages[-1].content)

Title: An Overview and Case Study of the Clinical AI Model Development Life Cycle for Healthcare Systems
Summary: Healthcare is one of the most promising areas for machine learning models to make a positive impact. However, successful adoption of AI-based systems in healthcare depends on engaging and educating stakeholders from diverse backgrounds about the development process of AI models. We present a broadly accessible overview of the development life cycle of clinical AI models that is general enough to be adapted to most machine learning projects, and then give an in-depth case study of the development process of a deep learning based system to detect aortic aneurysms in Computed Tomography (CT) exams. We hope other healthcare institutions and clinical practitioners find the insights we share about the development process useful in informing their own model development efforts and to increase the likelihood of successful deployment and integration of AI in healthcare.

Title: Fait

In [7]:
insight_agent = "Reads the papers and summarizes key ideas and insights."
SYSTEM_MESSAGE = """Role: You are an insight synthesizer agent.
Task: your job is to extract meaningful insights and highlight the main trends or findings across multiple papers. You should provide concise summary in 3 to 5 bullet points.
OUTPUT FORMAT:
Key Insights:
- <insight 1>
- <insight 2>
"""
insightAgent = AssistantAgent(name="InsightAgent",
                            description = agent_description,
                            system_message = SYSTEM_MESSAGE,
                            model_client = client
                            )



In [ ]:
insight_result = await insightAgent.run(task=paper_result.messages[-1].content)
print(insight_result.messages[-1].content)

Key Insights:

- Successful adoption of AI in healthcare requires education and engagement of diverse stakeholders, highlighting the importance of a comprehensible AI model development life cycle and deployment case studies, such as deep learning systems for detecting medical conditions like aortic aneurysms.

- Perceived authority of AI in predictive roles influences human decision-making significantly, often leading individuals to alter their reasoning about future behaviors, as observed in a study illustrating that reliance on AI predictions can result in irrational decision outcomes and financial losses.

- Generative AI models are revolutionizing information access systems by enabling advanced content creation (information generation) and methodical integration of existing data (information synthesis). These capabilities enhance user experiences and support the provision of accurate responses, showcasing the foundational transformation these models bring to IA technologies.

- Pri

In [8]:
report_agent = "Creates a comprehensive report based on the insights."
SYSTEM_MESSAGE = """Role: You are a report creator agent.
Task: your job is to create a comprehensive report based on the insights extracted from multiple papers. You should provide a well-structured summary that highlights the main findings and their implications.
OUTPUT FORMAT:
##Research Report
### Introduction
...
### Key Findings
...
### Discussion
...
### Conclusion
...
"""
reportAgent = AssistantAgent(name="ReportAgent",
                            description = agent_description,
                            system_message = SYSTEM_MESSAGE,
                            model_client = client
                            )



In [ ]:
report_result = await reportAgent.run(task=insight_result.messages[-1].content)
print(report_result.messages[-1].content)

## Research Report
### Introduction
The advancement of artificial intelligence (AI) in various sectors promises transformative capabilities, particularly in healthcare. This report synthesizes insights from recent studies to highlight the essential components for successful AI adoption, the impact of AI on decision-making, the evolution of information access through generative AI models, privacy considerations in healthcare machine learning (ML) applications, and the importance of robust security architectures.

### Key Findings
1. **Adoption of AI in Healthcare**:
   Successful integration of AI into healthcare hinges on educating and engaging diverse stakeholders. A comprehensible development life cycle for AI models is essential, as evidenced by the efficacy of deep learning systems in detecting conditions like aortic aneurysms.

2. **AI’s Influence on Decision-Making**:
   The perceived authority of AI in predictive analytics significantly affects human decision-making processes. S

In [9]:
report_agent = "Creates research gap analysis."
SYSTEM_MESSAGE = """Role: You are a report gap analysis agent.
Task: Identify 2–3 research gaps and 2–3 future directions.

    OUTPUT FORMAT:
    Research Gaps:
    - <Gap 1>
    - <Gap 2>

    Future Directions:
    - <Direction 1>
    - <Direction 2>

    FINAL REPORT
"""
gapAnalysisAgent = AssistantAgent(name="GapAnalysisAgent",
                            description = agent_description,
                            system_message = SYSTEM_MESSAGE,
                            model_client = client
                            )


In [ ]:

gapAnalysis_result = await gapAnalysisAgent.run(task=report_result.messages[-1].content)
print(gapAnalysis_result.messages[-1].content)

Research Gaps:
- The report highlights the importance of privacy-preserving machine learning in healthcare but does not delve deeply into specific techniques or challenges associated with ensuring privacy during the deployment of these systems. More research is needed to explore the effectiveness of various privacy-preserving methodologies in real-world applications.
- The discussion on AI’s influence on decision-making emphasizes the potential for irrational decisions due to overreliance on AI; however, it does not provide detailed insights into how existing AI systems can be adjusted to better complement human intuition. A gap exists in understanding the nuanced integration of AI insights with human decision-making processes.

Future Directions:
- Future research should focus on developing and testing advanced privacy-preserving techniques specific to healthcare data, ensuring these models effectively protect sensitive information while maintaining high performance. This could includ

In [13]:
textMentionTermination = TextMentionTermination("FINAL REPORT")
maxMessageTermination = MaxMessageTermination(10)
termination = textMentionTermination | maxMessageTermination

Create a round robin chat group 

In [23]:
agentList = [topicAgent, paperAgent, insightAgent, reportAgent, gapAnalysisAgent]
roundRobinGroupChat = RoundRobinGroupChat(participants=agentList, termination_condition=termination)

#Console(stream=task).run(roundRobinGroupChat)

await Console(roundRobinGroupChat.run_stream(task=task))

---------- TextMessage (user) ----------
AI in healthcare
---------- TextMessage (TopicAgent) ----------
Refined Research Topic: "Development and Deployment of AI Models for Detecting Aortic Aneurysms in Computed Tomography: Challenges and Best Practices"

TASK_COMPLETED_TOPIC_AGENT
---------- TextMessage (PaperAgent) ----------
Here are some relevant research papers on the topic of AI in healthcare:

1. **An Overview and Case Study of the Clinical AI Model Development Life Cycle for Healthcare Systems**
   - This paper provides an accessible overview of the development life cycle of clinical AI models, with a case study focused on developing a deep learning system to detect aortic aneurysms in CT exams. It aims to help healthcare institutions adopt AI successfully.

2. **Faith in AI can narrow the futures individuals consider**
   - This study explores how AI predictions influence human decision-making, especially in contexts where AI is perceived as a predictive authority. It has imp

TaskResult(messages=[TextMessage(id='54968b7a-5ffa-4e9c-8a29-09a6420758d6', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 10, 2, 18, 25, 20, 297768, tzinfo=datetime.timezone.utc), content='AI in healthcare', type='TextMessage'), TextMessage(id='b5c2c163-5dd6-4b53-9191-24c09fa4bdfe', source='TopicAgent', models_usage=RequestUsage(prompt_tokens=798, completion_tokens=38), metadata={}, created_at=datetime.datetime(2026, 10, 2, 18, 25, 21, 976632, tzinfo=datetime.timezone.utc), content='Refined Research Topic: "Development and Deployment of AI Models for Detecting Aortic Aneurysms in Computed Tomography: Challenges and Best Practices"\n\nTASK_COMPLETED_TOPIC_AGENT', type='TextMessage'), TextMessage(id='ff90ddf4-c637-48f3-8a34-9ef03466a16b', source='PaperAgent', models_usage=RequestUsage(prompt_tokens=1295, completion_tokens=304), metadata={}, created_at=datetime.datetime(2026, 10, 2, 18, 25, 25, 328246, tzinfo=datetime.timezone.utc), content='Here are so

Create UserProxyAgent for manual approval

In [10]:
userProxyAgent = UserProxyAgent(
    "UserProxyAgent",
    description="A proxy for the user to approve or disapprove tasks."
)

user_approved = ""


In [11]:
"""async def selector_func_with_user_proxy(messages):
    topic_result = await topicAgent.run(task=task)
    user_approved = await user_proxy_agent.run(task=task)
    return topic_result, user_approved
"""

# Track whether the user has already approved once
user_approved = False

def selector_func_with_user_proxy(messages: Sequence[BaseAgentEvent | BaseChatMessage]) -> str | None:
    global user_approved

    # If first message, start with topic agent
    if len(messages) == 0:
        return topicAgent.name

    last_source = messages[-1].source

    # If topic_agent just finished, ask for human approval ONCE
    if last_source == topicAgent.name and not user_approved:
        return userProxyAgent.name

    # If user just responded (approve or reject)
    if last_source == userProxyAgent.name:
        content = messages[-1].content.upper()
        if "APPROVE" in content:
            user_approved = True
            # Proceed to the next logical agent (e.g., paper_agent)
            return paperAgent.name
        else:
            # If disapproved, return to topic_agent for revision
            return topicAgent.name

    # After approval, let the rest of the agents talk automatically
    if user_approved:
        return None  # means use default sequencing in SelectorGroupChat

    return None

In [14]:
SELECTOR_PROMPT = """You are a selector agent that decides which agent should speak next in a multi-agent conversation.
You have access to the following agents:
- TopicAgent: Refines a broad research idea into a specific topic.
- PaperAgent: Searches for relevant papers based on a research topic.
- InsightAgent: Reads the papers and summarizes key ideas and insights.
- ReportAgent: Creates a comprehensive report based on the insights.
- GapAnalysisAgent: Creates research gap analysis.
Your task is to determine which agent should speak next based on the conversation history and the current state of the research process. You should consider the following:
- If the topic has not been refined yet, TopicAgent should speak next.
- If the topic has been refined and user approval is needed, UserProxyAgent should speak next.
- If the topic has been refined and user approval has been granted, PaperAgent should speak next.
- If the papers have been searched and insights need to be summarized, InsightAgent should speak next.
- If the insights have been summarized and a report needs to be created, ReportAgent should speak next.
- If the report has been created and a gap analysis needs to be done, GapAnalysisAgent should speak next.
- If the gap analysis has been completed, the conversation can end.
You should also consider the content of the messages and any relevant information provided by the agents. Your goal is to facilitate a smooth and logical flow of the conversation, ensuring that each agent has the opportunity to contribute at the appropriate time. You should also be mindful of the user's input and any approvals or disapprovals they provide. Your decisions should be based on the current state of the research process and the information available in the conversation history. You should aim to create a coherent and productive dialogue that leads to a successful research outcome.
Your output should be the name of the agent that should speak next, or None if the default sequencing in SelectorGroupChat should be used. You should also provide a brief explanation for your decision, highlighting the reasoning behind your choice and any relevant context from the conversation history. Your explanation should be concise and informative, helping to clarify the rationale for your selection and ensuring that all agents understand the flow of the conversation. You should also be prepared to adjust your decisions based on new information or changes in the conversation, demonstrating flexibility and adaptability in your role as a selector agent.
"""

task = "AI in healthcare"
agentList = [topicAgent, paperAgent, insightAgent, reportAgent, gapAnalysisAgent, userProxyAgent]
selectorGroupChat = SelectorGroupChat(participants=agentList, selector_func=selector_func_with_user_proxy, termination_condition=termination, selector_prompt=SELECTOR_PROMPT, allow_selector_to_speak=False)

TypeError: SelectorGroupChat.__init__() got an unexpected keyword argument 'allow_selector_to_speak'